# 02f · D0: the weights currently deployed in the PlantIQ backend

`backend/data/models/best_resnet50_coffee.pth` (read only, never changed) is rebuilt exactly as
`backend/app/services/cnn.py` does: ResNet-50, `Dropout(0.4)+Linear` head, 224x224, ImageNet normalisation.

**Caveat for the paper:** we do not know which split these weights were trained on. If it was a random file
split of this dataset (likely, see `PLAN.md` §0.2), they have already seen copies of almost every test image, so
their score here is optimistic and not comparable. D0 is reported for reference only; the fair comparison is
R0 vs V2 vs G1 vs Y1. D0's CPU latency is the reference for the replacement rule. (~2 min, no training.)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import hashlib

import torch

import pandas as pd

from common import config, data, env
from common.cv import folds_to_run, preds_path
from common.metrics import classification_metrics
from common.train import load_deployed, measure_cost, predict, save_preds

config.ensure_dirs()
device = env.require_cuda()
uniq, splits = data.load_unique(), data.load_splits()
images = data.load_images(uniq)
sha = hashlib.sha256(config.DEPLOYED_WEIGHTS.read_bytes()).hexdigest()
print("deployed weights SHA-256:", sha)
model = load_deployed().to(device).eval()
tf = data.build_transform("eval", 224)

In [ ]:
rows = {}
for k in folds_to_run():
    idx = data.fold_indices(uniq, splits, k)
    for split in ("val", "test"):
        loader = data.make_loader(images, uniq, idx[split], tf, batch_size=128, train=False, workers=0)
        lg, y, j = predict(model, loader, device)
        save_preds(preds_path("D0", k, split), uniq, lg, y, j)
        if split == "test":
            rows[k] = classification_metrics(y, lg.argmax(1))
            print(f"[D0] fold {k} TEST acc {rows[k]['accuracy']:.4f} macro-F1 {rows[k]['macro_f1']:.4f}")

In [ ]:
env.write_json(config.CNN_RESULTS / "D0_summary.json",
               {"model": "D0", "weights": str(config.DEPLOYED_WEIGHTS.relative_to(config.REPO)), "sha256": sha,
                "folds": {k: {"test": v} for k, v in rows.items()}, "meta": env.run_meta(),
                "caveat": "training split unknown; likely saw copies of the test images"})
cost = measure_cost(model, 224, "D0")
env.write_json(config.CNN_RESULTS / "D0_cost.json", cost)
print(cost)
pd.DataFrame({k: {"test_acc": v["accuracy"], "test_macro_f1": v["macro_f1"]} for k, v in rows.items()}).T